---

<p align="center">
   <img src="img/card_vasc.jpg" width="1000" height="300"> 
</p>

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">


<h1 align="center"><b> 🖤 Mortalidad Cardiovascular Semanal Promedio </b></h1>

---

Los datos descritos en **cmort**, de la librería **astsa**, son series extraídas de un estudio de Shumway sobre los posibles efectos de la contaminación en la mortalidad cardiovascular semanal promedio en el condado de Los Ángeles a lo largo de los 10 años.Hay 508 promedios suavizados de seis días obtenidos al filtrar los valores diarios durante el período de 10 años 1970 - 1979.

</div>

---

## **Librerías y Carga de los Datos**

In [ ]:
# Instalar Librerías #
install.packages("astsa")
install.packages("quantmod")
install.packages("tseries")
install.packages("timeSeries")
install.packages("forecast")
install.packages("xts")
install.packages("TSA")
install.packages("lmtest")
install.packages("nortest")
install.packages("car")
install.packages("urca")
install.packages("dynlm")
install.packages("MLmetrics")
install.packages("ggplot2")

In [ ]:
# Cargar Librerías #
library(astsa) # Para acceder a la serie 'cmort'
library(quantmod)
library(tseries)  # Para adf.test
library(timeSeries)
library(forecast)
library(xts)
library(TSA)
library(lmtest)
library(nortest)
library(car)      # Para leveneTest
library(urca)
library(dynlm)
library(MLmetrics)
library(ggplot2)

## **Análisis Exploratorio**

In [ ]:
# Cargar los datos #
options(repr.plot.width=14, repr.plot.height=8)
ts.plot(cmort, main= "Serie de Mortalidad Cardiovascular Semanal Promedio", lwd=2, ylab="Mortalidad por 100,000 habitantes", xlab="Año"); grid(col="lightgray")
abline(v=c(seq(1971,1979)), col="blue")

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

### **Prueba de Estacionaridad**

La prueba de Dickey-Fuller es una prueba de raíz unitaria que se utiliza comúnmente para determinar si una serie de tiempo es estacionaria o no.     

$H_0$ : No Estacionaria $\hspace{0.4cm}$ $vs$  $\hspace{0.4cm}$ $H_a$ : Estacionaria

</div>

In [ ]:
# Aplicar la prueba de Dickey-Fuller de Estacionaridad #
resultado_adf <- suppressWarnings(adf.test(cmort))

print(resultado_adf)

if (resultado_adf$p.value < 0.01) {
  print("La serie es estacionaria")
} else {
  print("La serie no es estacionaria")
}

---

## **Identificación**

### **Funciones de Autocorrelación y Autocorrelación Parcial**

In [ ]:
# 1. Parámetros e intervalos de confianza
n <- length(cmort)
lags <- 30
ci <- qnorm((1 + 0.95) / 2) / sqrt(n) # Límite al 95% de confianza

# 2. Calcular ACF y PACF
acf_obj  <- acf(cmort, lag.max = lags, plot = FALSE)
pacf_obj <- pacf(cmort, lag.max = lags, plot = FALSE)

# Extraer valores y asegurar que coincida la longitud de rezagos (omitiendo lag 0 en ACF)
acf_vals  <- as.numeric(acf_obj$acf[-1])
pacf_vals <- as.numeric(pacf_obj$acf)

n_lags_acf  <- length(acf_vals)
n_lags_pacf <- length(pacf_vals)

# 3. Construir data.frame unificado dinámicamente
df_acf <- data.frame(
  Panel = "Función de Autocorrelación (ACF)",
  Lag = 1:n_lags_acf,
  Valor = acf_vals
)

df_pacf <- data.frame(
  Panel = "Función de Autocorrelación Parcial (PACF)",
  Lag = 1:n_lags_pacf,
  Valor = pacf_vals
)

df_autocorr <- rbind(df_acf, df_pacf)

# Data frame para las bandas de confianza
df_ci <- data.frame(
  Panel = rep(c("Función de Autocorrelación (ACF)", "Función de Autocorrelación Parcial (PACF)"), each = 2),
  ci = c(ci, -ci, ci, -ci)
)

# 4. Gráfica en ggplot2
ggplot(df_autocorr, aes(x = Lag, y = Valor)) +
  # Segmentos verticales y puntos (Formato Needle/Lollipop)
  geom_segment(aes(xend = Lag, yend = 0), color = "gray30") +
  geom_point(color = "steelblue", size = 2) +
  
  # Línea horizontal base en y = 0
  geom_hline(yintercept = 0, color = "gray50") +
  
  # Líneas de intervalos de confianza al 95%
  geom_hline(data = df_ci, aes(yintercept = ci), linetype = "dashed", color = "firebrick") +
  
  # Disposición en 2 paneles superpuestos
  facet_wrap(~ Panel, ncol = 1, scales = "free_y") +
  
  labs(
    title = "Diagnóstico de Autocorrelación para 'cmort'",
    x = "Rezago (Lag)",
    y = ""
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5),
    strip.text = element_text(face = "bold", size = 11),
    panel.grid.minor = element_blank()
  )


In [ ]:
# 1. Calcular el filtro de media móvil (sides = 2, ventana de 3)
fma_cmort <- stats::filter(cmort, sides = 2, filter = c(1, 1, 1) / 3)

# 2. Extraer el eje de tiempo en años
tiempo <- as.numeric(time(cmort))

# 3. Construir un data.frame unificado para ggplot2
df_original <- data.frame(
  Panel = "Original",
  Tiempo = tiempo,
  Mortalidad = as.numeric(cmort)
)

df_suavizada <- data.frame(
  Panel = "Suavizada",
  Tiempo = tiempo,
  Mortalidad = as.numeric(fma_cmort)
)

df_mortalidad <- rbind(df_original, df_suavizada)

# Asegurar el orden de los paneles (Original arriba, Suavizada abajo)
df_mortalidad$Panel <- factor(df_mortalidad$Panel, levels = c("Original", "Suavizada"))

# Data frame para las líneas verticales de referencia anual
df_lines <- data.frame(
  Panel = factor(rep(c("Original", "Suavizada"), each = 9), levels = c("Original", "Suavizada")),
  Año = rep(1971:1979, times = 2),
  Color = rep(c("firebrick", "steelblue"), each = 9)
)

# 4. Graficar en ggplot2 con facet_wrap
ggplot(df_mortalidad, aes(x = Tiempo, y = Mortalidad)) +
  # Líneas verticales anuales (1971 a 1979)
  geom_vline(
    data = df_lines, 
    aes(xintercept = Año, color = Color), 
    linetype = "dashed", 
    linewidth = 0.6, 
    alpha = 0.7
  ) +
  
  # Serie temporal principal
  geom_line(color = "gray20", linewidth = 0.7, na.rm = TRUE) +
  
  # Asignación manual de colores para las líneas verticales
  scale_color_identity() +
  scale_x_continuous(breaks = seq(1970, 1980, by = 1)) +
  
  # Paneles superpuestos
  facet_wrap(~ Panel, ncol = 1, scales = "free_y") +
  
  labs(
    title = "Análisis de Estacionalidad: Serie Original vs. Suavizada",
    subtitle = "Mortalidad cardiovascular semanal promedio (Filtro de media móvil de 3 puntos)",
    x = "Año",
    y = "Mortalidad Cardiovascular"
  ) +
  
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5),
    plot.subtitle = element_text(size = 11, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 12),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(color = "gray92")
  )


In [ ]:
# Descomposición muestra una tendencia general decreciente #

# 1. Aplicar filtro de media móvil y descomposición
fma_cmort <- stats::filter(cmort, sides = 2, filter = c(1, 1, 1) / 3)
descomp <- decompose(fma_cmort)

# 2. Extraer tiempo numérico (Años)
tiempo <- as.numeric(time(cmort))

# 3. Crear data frame unificado con los 4 componentes
df_descomp <- data.frame(
  Tiempo = rep(tiempo, 4),
  Valor = c(
    as.numeric(descomp$x),
    as.numeric(descomp$trend),
    as.numeric(descomp$seasonal),
    as.numeric(descomp$random)
  ),
  Componente = factor(
    rep(c("Serie Filtrada (Observada)", "Tendencia", "Estacionalidad", "Residuos (Aleatorio)"), each = length(tiempo)),
    levels = c("Serie Filtrada (Observada)", "Tendencia", "Estacionalidad", "Residuos (Aleatorio)")
  )
)

# 4. Gráfica en ggplot2
ggplot(df_descomp, aes(x = Tiempo, y = Valor)) +
  # Línea horizontal de referencia en 0 para los componentes estacional y aleatorio
  geom_hline(data = subset(df_descomp, Componente %in% c("Estacionalidad", "Residuos (Aleatorio)")), 
             aes(yintercept = 0), color = "gray60", linetype = "dashed") +
  
  # Serie de tiempo de cada componente
  geom_line(color = "steelblue", linewidth = 0.7, na.rm = TRUE) +
  
  # Paneles apilados en 4 filas con escala y libre
  facet_wrap(~ Componente, ncol = 1, scales = "free_y", strip.position = "right") +
  
  scale_x_continuous(breaks = seq(1970, 1980, by = 1)) +
  
  labs(
    title = "Descomposición Clásica de la Serie de Mortalidad Cardiovascular",
    subtitle = "Serie suavizada mediante filtro de media móvil (3 puntos)",
    x = "Año",
    y = ""
  ) +
  
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5),
    plot.subtitle = element_text(size = 11, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 10),
    strip.background = element_rect(fill = "gray95", color = NA),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(color = "gray92")
  )


In [ ]:
# El perfil estacional, evidencia una correlación entre periodos #
par(mfrow=c(2,1))
monthplot(cmort, main="Promedio de mortalidad semanal")
seasonplot(cmort, main="Perfil estacional")

# Un perfil estacional es como poner todos los años uno encima del otro para ver qué partes del año se comportan igual y cuáles no #

In [ ]:
# 1. Extraer información de tiempo y frecuencia de 'cmort'
n_obs <- length(cmort)
tiempo <- as.numeric(time(cmort))
años <- floor(tiempo)
# Calcular la semana del año (1 a 52)
semanas <- round((tiempo - años) * frequency(cmort)) + 1

df_base <- data.frame(
  Año = factor(años),
  Semana = semanas,
  Mortalidad = as.numeric(cmort)
)

# 2. PANEL 1: Promedio por Semana (Monthplot equivalente)
# Calcular promedio de mortalidad por cada semana del año a lo largo de todos los años
df_promedios <- aggregate(Mortalidad ~ Semana, data = df_base, FUN = mean)

df_panel1 <- data.frame(
  Panel = "Promedio de Mortalidad Semanal (Ciclo Anual)",
  Semana = df_promedios$Semana,
  Mortalidad = df_promedios$Mortalidad,
  Año = NA
)

# 3. PANEL 2: Perfil Estacional por Año (Seasonplot equivalente)
df_panel2 <- data.frame(
  Panel = "Perfil Estacional por Año",
  Semana = df_base$Semana,
  Mortalidad = df_base$Mortalidad,
  Año = df_base$Año
)

# 4. Graficar el Panel 1 (Promedio semanal)
g1 <- ggplot(df_panel1, aes(x = Semana, y = Mortalidad)) +
  geom_line(color = "firebrick", linewidth = 1) +
  geom_point(color = "firebrick", size = 1.8) +
  geom_hline(yintercept = mean(cmort), linetype = "dashed", color = "gray40") +
  labs(
    title = "Promedio de Mortalidad Semanal (Ciclo Anual)",
    x = "Semana del Año",
    y = "Mortalidad"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 12, hjust = 0.5),
    panel.grid.minor = element_blank()
  )

# 5. Graficar el Panel 2 (Líneas por cada año)
g2 <- ggplot(df_panel2, aes(x = Semana, y = Mortalidad, color = Año, group = Año)) +
  geom_line(linewidth = 0.7, alpha = 0.8) +
  scale_color_viridis_d(option = "turbo") +
  labs(
    title = "Perfil Estacional por Año",
    x = "Semana del Año",
    y = "Mortalidad",
    color = "Año"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 12, hjust = 0.5),
    legend.position = "right",
    panel.grid.minor = element_blank()
  )

# 6. Unificar mediante combinación de datos usando facet_wrap
# Preparar data frame unificado
df_panel1_plot <- data.frame(
  Panel = "1. Promedio de Mortalidad Semanal (Ciclo Anual)",
  Semana = df_promedios$Semana,
  Mortalidad = df_promedios$Mortalidad,
  Grupo = "Promedio General"
)

df_panel2_plot <- data.frame(
  Panel = "2. Perfil Estacional Comparativo por Año",
  Semana = df_base$Semana,
  Mortalidad = df_base$Mortalidad,
  Grupo = as.character(df_base$Año)
)

df_total <- rbind(df_panel1_plot, df_panel2_plot)

ggplot(df_total, aes(x = Semana, y = Mortalidad, group = Grupo, color = Grupo)) +
  geom_line(linewidth = 0.7) +
  facet_wrap(~ Panel, ncol = 1, scales = "free_y") +
  scale_x_continuous(breaks = seq(1, 52, by = 4)) +
  labs(
    title = "Análisis del Perfil Estacional de Mortalidad Cardiovascular",
    x = "Semana del Año (1 a 52)",
    y = "Mortalidad por 100,000 hab.",
    color = "Año / Grupo"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5),
    strip.text = element_text(face = "bold", size = 11),
    legend.position = "right",
    panel.grid.minor = element_blank()
  )


---

## **Funciones de Autocorrelación y Autocorrelación Parcial**

In [ ]:
# Análisis de la ACF y PACF de la Serie Original #

# 1. Parámetros de la serie e intervalo de confianza (95%)
n <- length(cmort)
lags_max <- 200
ci <- qnorm((1 + 0.95) / 2) / sqrt(n) # Límite al 95% de confianza

# 2. Calcular ACF y PACF hasta 200 rezagos
acf_obj  <- acf(cmort, lag.max = lags_max, plot = FALSE)
pacf_obj <- pacf(cmort, lag.max = lags_max, plot = FALSE)

# Extraer valores como vectores numéricos (omitiendo lag 0 en ACF)
acf_vals  <- as.numeric(acf_obj$acf[-1])
pacf_vals <- as.numeric(pacf_obj$acf)

n_lags_acf  <- length(acf_vals)
n_lags_pacf <- length(pacf_vals)

# 3. Construir data.frame unificado
df_acf <- data.frame(
  Panel = "Función de Autocorrelación (ACF)",
  Lag = 1:n_lags_acf,
  Valor = acf_vals
)

df_pacf <- data.frame(
  Panel = "Función de Autocorrelación Parcial (PACF)",
  Lag = 1:n_lags_pacf,
  Valor = pacf_vals
)

df_autocorr <- rbind(df_acf, df_pacf)
df_autocorr$Panel <- factor(df_autocorr$Panel, 
                            levels = c("Función de Autocorrelación (ACF)", 
                                       "Función de Autocorrelación Parcial (PACF)"))

# Data frame para las bandas de confianza de Quenouille
df_ci <- data.frame(
  Panel = factor(rep(c("Función de Autocorrelación (ACF)", 
                       "Función de Autocorrelación Parcial (PACF)"), 
                     each = 2),
                 levels = c("Función de Autocorrelación (ACF)", 
                            "Función de Autocorrelación Parcial (PACF)")),
  ci = c(ci, -ci, ci, -ci)
)

# 4. Gráfica en ggplot2
ggplot(df_autocorr, aes(x = Lag, y = Valor)) +
  # Segmentos verticales delgados para alta densidad de lags (200)
  geom_segment(aes(xend = Lag, yend = 0), color = "steelblue", linewidth = 0.4) +
  
  # Línea base horizontal en y = 0
  geom_hline(yintercept = 0, color = "gray40", linewidth = 0.5) +
  
  # Intervalos de confianza al 95%
  geom_hline(data = df_ci, aes(yintercept = ci), linetype = "dashed", color = "firebrick") +
  
  # Configurar marcas en el eje X para destacar ciclos anuales (cada 52 rezagos)
  scale_x_continuous(breaks = c(1, seq(52, lags_max, by = 52))) +
  
  # Disposición en 2 paneles apilados
  facet_wrap(~ Panel, ncol = 1, scales = "free_y") +
  
  labs(
    title = "Diagnóstico de Autocorrelación Extendida (Lag Max = 200)",
    subtitle = "Serie de Mortalidad Cardiovascular ('cmort')",
    x = "Rezago (Lag en semanas)",
    y = ""
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5),
    plot.subtitle = element_text(size = 11, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 11),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_line(color = "gray92")
  )

In [ ]:
# Análisis de la ACF y PACF de la Primera Diferencia #

# 1. Calcular la primera diferencia de la serie
diff_cmort <- diff(cmort)

# 2. Parámetros e intervalo de confianza (95%)
n <- length(diff_cmort)
lags_max <- 200
ci <- qnorm((1 + 0.95) / 2) / sqrt(n) # Límite al 95% de confianza

# 3. Calcular ACF y PACF hasta 200 rezagos
acf_obj  <- acf(diff_cmort, lag.max = lags_max, plot = FALSE)
pacf_obj <- pacf(diff_cmort, lag.max = lags_max, plot = FALSE)

# Extraer valores numéricos (omitiendo lag 0 en ACF)
acf_vals  <- as.numeric(acf_obj$acf)
pacf_vals <- as.numeric(pacf_obj$acf)

n_lags_acf  <- length(acf_vals)
n_lags_pacf <- length(pacf_vals)

# 4. Construir data.frame unificado
df_acf <- data.frame(
  Panel = "ACF de la Primera Diferencia",
  Lag = 1:n_lags_acf,
  Valor = acf_vals
)

df_pacf <- data.frame(
  Panel = "PACF de la Primera Diferencia",
  Lag = 1:n_lags_pacf,
  Valor = pacf_vals
)

df_autocorr <- rbind(df_acf, df_pacf)
df_autocorr$Panel <- factor(df_autocorr$Panel, 
                            levels = c("ACF de la Primera Diferencia", 
                                       "PACF de la Primera Diferencia"))

# Data frame para las bandas de confianza de Quenouille
df_ci <- data.frame(
  Panel = factor(rep(c("ACF de la Primera Diferencia", 
                       "PACF de la Primera Diferencia"), 
                     each = 2),
                 levels = c("ACF de la Primera Diferencia", 
                            "PACF de la Primera Diferencia")),
  ci = c(ci, -ci, ci, -ci)
)

# 5. Graficar en ggplot2
ggplot(df_autocorr, aes(x = Lag, y = Valor)) +
  # Segmentos delgados para alta densidad de rezagos
  geom_segment(aes(xend = Lag, yend = 0), color = "steelblue", linewidth = 0.4) +
  
  # Línea base en y = 0
  geom_hline(yintercept = 0, color = "gray40", linewidth = 0.5) +
  
  # Intervalos de confianza al 95%
  geom_hline(data = df_ci, aes(yintercept = ci), linetype = "dashed", color = "firebrick") +
  
  # Marcas del eje X cada 52 semanas (ciclo anual)
  scale_x_continuous(breaks = c(1, seq(52, lags_max, by = 52))) +
  
  # Disposición en 2 paneles superpuestos
  facet_wrap(~ Panel, ncol = 1, scales = "free_y") +
  
  labs(
    title = "Diagnóstico de Autocorrelación - Serie Diferenciada (d = 1)",
    subtitle = "Primera Diferencia de la Mortalidad Cardiovascular (diff(cmort))",
    x = "Rezago (Lag en semanas)",
    y = ""
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5),
    plot.subtitle = element_text(size = 11, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 11),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_line(color = "gray92")
  )


In [ ]:
#  Análisis de la ACF y PACF de la Primera Diferencia Estacional con S=52 #
# Parece que aplicar una sola diferencia es adecuado, sin embargo,  #
# se considera una diferencia ordinaria y una diferencia estacional #
# de rezago 52 es decir, (1-B)(1-B^{52}).  

# 1. Aplicar diferencia ordinaria y estacional (1-B)(1-B^52)
diff_estacional_cmort <- diff(diff(cmort), lag = 52)

# 2. Parámetros e intervalo de confianza al 95%
n <- length(diff_estacional_cmort)
lags_max <- 200
ci <- qnorm((1 + 0.95) / 2) / sqrt(n) # Límite al 95% de confianza

# 3. Calcular ACF y PACF hasta 200 rezagos
acf_obj  <- acf(diff_estacional_cmort, lag.max = lags_max, plot = FALSE)
pacf_obj <- pacf(diff_estacional_cmort, lag.max = lags_max, plot = FALSE)

# Extraer valores numéricos (omitiendo lag 0 en la ACF)
acf_vals  <- as.numeric(acf_obj$acf)
pacf_vals <- as.numeric(pacf_obj$acf)

n_lags_acf  <- length(acf_vals)
n_lags_pacf <- length(pacf_vals)

# 4. Construir data.frame unificado
df_acf <- data.frame(
  Panel = "ACF: (1-B)(1-B^52) cmort",
  Lag = 1:n_lags_acf,
  Valor = acf_vals
)

df_pacf <- data.frame(
  Panel = "PACF: (1-B)(1-B^52) cmort",
  Lag = 1:n_lags_pacf,
  Valor = pacf_vals
)

df_autocorr <- rbind(df_acf, df_pacf)
df_autocorr$Panel <- factor(
  df_autocorr$Panel, 
  levels = c("ACF: (1-B)(1-B^52) cmort", "PACF: (1-B)(1-B^52) cmort")
)

# Data frame para las bandas de confianza de Quenouille
df_ci <- data.frame(
  Panel = factor(
    rep(c("ACF: (1-B)(1-B^52) cmort", "PACF: (1-B)(1-B^52) cmort"), each = 2),
    levels = c("ACF: (1-B)(1-B^52) cmort", "PACF: (1-B)(1-B^52) cmort")
  ),
  ci = c(ci, -ci, ci, -ci)
)

# 5. Graficar en ggplot2
ggplot(df_autocorr, aes(x = Lag, y = Valor)) +
  # Segmentos delgados para evitar saturación visual a 200 rezagos
  geom_segment(aes(xend = Lag, yend = 0), color = "steelblue", linewidth = 0.4) +
  
  # Línea base en y = 0
  geom_hline(yintercept = 0, color = "gray40", linewidth = 0.5) +
  
  # Intervalos de confianza al 95%
  geom_hline(data = df_ci, aes(yintercept = ci), linetype = "dashed", color = "firebrick") +
  
  # Marcas en el eje X destacando los períodos de 52 semanas
  scale_x_continuous(breaks = c(1, seq(52, lags_max, by = 52))) +
  
  # Paneles apilados verticalmente
  facet_wrap(~ Panel, ncol = 1, scales = "free_y") +
  
  labs(
    title = "Diagnóstico de Autocorrelación con Diferencia Ordinaria y Estacional",
    subtitle = "Transformación: (1 - B)(1 - B52) sobre la Mortalidad Cardiovascular ('cmort')",
    x = "Rezago (Lag en semanas)",
    y = ""
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5),
    plot.subtitle = element_text(size = 11, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 11),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_line(color = "gray92")
  )

In [ ]:
# La ACF y PACF de la serie con diferencia ordinaria y una diferencia              #
# estacional de rezago 52, indican que un modelo                                   #
# $ARIMA(1,1,0)x(0,1,1)52$ podría ser adecuado para predecir la                  #
# cantidad de muertes por enfermedad cardiaca debido a la contaminación ambiental  #

par(mfrow=c(2,1))
plot(diff(cmort), main="Diferencia Ordinaria")
plot(diff(diff(cmort),lag=52), main="Diferencia Ordinaria y Estacional")

In [ ]:
# 1. Calcular las dos transformaciones de diferenciación
diff_ord <- diff(cmort)
diff_ord_est <- diff(diff(cmort), lag = 52)

# 2. Extraer los vectores de tiempo exactos para cada serie
tiempo_ord <- as.numeric(time(diff_ord))
tiempo_ord_est <- as.numeric(time(diff_ord_est))

# 3. Construir el data.frame unificado
df_ord <- data.frame(
  Panel = "1. Diferencia Ordinaria: (1 - B) cmort",
  Tiempo = tiempo_ord,
  Valor = as.numeric(diff_ord)
)

df_ord_est <- data.frame(
  Panel = "2. Diferencia Ordinaria y Estacional: (1 - B)(1 - B^52) cmort",
  Tiempo = tiempo_ord_est,
  Valor = as.numeric(diff_ord_est)
)

df_diferencias <- rbind(df_ord, df_ord_est)

# Fijar el orden estricto de los paneles
df_diferencias$Panel <- factor(
  df_diferencias$Panel,
  levels = c(
    "1. Diferencia Ordinaria: (1 - B) cmort",
    "2. Diferencia Ordinaria y Estacional: (1 - B)(1 - B^52) cmort"
  )
)

# 4. Graficar en ggplot2
ggplot(df_diferencias, aes(x = Tiempo, y = Valor)) +
  # Línea horizontal de referencia en y = 0
  geom_hline(yintercept = 0, color = "firebrick", linetype = "dashed", linewidth = 0.6) +
  
  # Serie temporal diferenciada
  geom_line(color = "steelblue", linewidth = 0.6) +
  
  # Marcas en el eje X por años
  scale_x_continuous(breaks = seq(1970, 1980, by = 1)) +
  
  # Disposición en 2 paneles apilados
  facet_wrap(~ Panel, ncol = 1, scales = "free_y") +
  
  labs(
    title = "Comparación de Series Diferenciadas de Mortalidad Cardiovascular",
    subtitle = "Evaluación de Estacionariedad en Media y Remoción de Estacionalidad",
    x = "Año",
    y = "Diferencia de Mortalidad"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5),
    plot.subtitle = element_text(size = 11, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 11),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(color = "gray92")
  )


## **Búsqueda de los Órdenes**

In [ ]:
# Valores de orden
p <- 1; q <- 1; P <- 1; Q <- 2

# Combinaciones posibles
combinaciones <- expand.grid(p = 0:p, q = 0:q, P = 0:P, Q = 0:Q)
n_modelos <- nrow(combinaciones)

# Inicializar matriz para resultados
ic_mod <- matrix(NA, nrow = n_modelos, ncol = 6)
colnames(ic_mod) <- c("p", "q", "P", "Q", "AIC", "BIC")

# Iterar sobre combinaciones
for (k in 1:n_modelos) {
  i <- combinaciones$p[k]
  j <- combinaciones$q[k]
  s <- combinaciones$P[k]
  m <- combinaciones$Q[k]
  
  # Ajustar modelo con manejo de errores
  modelo <- tryCatch({
    Arima(cmort, order = c(i, 1, j),
          seasonal = list(order = c(s, 1, m), period = 12),
          method = "ML")
  }, error = function(e) NULL)
  
  if (!is.null(modelo)) {
    ic_mod[k, ] <- c(i, j, s, m, AIC(modelo), BIC(modelo))
  }
}

# Convertir a data.frame y eliminar NAs (modelos no ajustados)
ic_mod_df <- as.data.frame(ic_mod) %>% na.omit()

# Es importante señalar la importancia de buscar el modelo con "cmort" y "cmort_train", con el fin de ver la Estabilidad de los órdenes #

In [ ]:
# Mostrar resultados ordenados por AIC
ic_mod_df[order(ic_mod_df$AIC), ]

In [ ]:
# Mostrar resultados ordenados por BIC
ic_mod_df[order(ic_mod_df$BIC), ]

## **Mejor Modelo Según Criterios de Información**

### **SARIMA** $(1,1,0)\times(0,1,1)_{52}$

---

### **Usando la función auto.arima()**

In [ ]:
# Ahora vamos a utilizar la función $auto.arima para calcular el modelo      #
# de forma automática usando el criterio AIC y sin considerar estacionalidad #
# para enfatizar la importancia de identificar S=52                          #
#?auto.arima
auto.arima(cmort, ic = "aic", seasonal = F)
# Como resultado, obtenemos que el mejor modelo es un $ARIMA(2,1,0)$, #
# por ende, vamos a ajustar ambos modelos propuestos. #

## **Estimación de Parámetros para Dos Modelos Tentativos**

### **Modelo 1 :** Usando los criterios y componente estacional

In [ ]:
# Usando criterios #
mod1 = arima(cmort, order = c(1, 1, 0), seasonal = list(order = c(0, 1, 1), period = 52),
           method = c("CSS-ML"))

### **Modelo 2 :** Usando la recomentación de auto.arima()

In [ ]:
# Usando auto.arima() #
mod2 = arima(cmort, order = c(2, 1, 1), method = c("CSS-ML"))

## **Diagnóstico Residual**

In [ ]:
# Ahora nos disponemos a evaluar el ajuste de los modelos #
resmod1=residuals(mod1)
resmod2=residuals(mod2)

In [ ]:
# Valores ajustados #
ajust1=cmort-resmod1
ajust2=cmort-resmod2

In [ ]:
# Gráfico para los valores ajustados #
par(mfrow=c(2,1))
# Modelo 1 #
ts.plot(cmort, ajust1 , main = "Valores ajustados con arima()")
lines(cmort, col="red")
# Modelo 2 #
ts.plot(cmort, ajust2 , main = "Valores ajustados con auto.arima()")
lines(cmort, col="blue")

## **Observación:** 

<div style="text-align: justify; line-height: 1.5; font-size: 18px;”> 
 
<h1 align="center"><b>  </b></h1>

Aunque el **Modelo 2** $\text{ARIMA}(2,1,1)$ presenta un rendimiento aparentemente superior en el ajuste inicial respecto al **Modelo 1** $\text{SARIMA}(1,1,0) \times (0,1,1)_{52}$, existe el riesgo de que este comportamiento responda a un **sobreajuste** (*overfitting*), afectando la precisión y eficiencia de los pronósticos fuera de muestra.  
  
Para seleccionar formalmente el modelo con mayor capacidad de generalización, resulta indispensable evaluar criterios estadísticos de selección. En primera instancia, se analizarán los criterios de información de Akaike (**AIC**) y Bayesiano (**BIC**). 

</div>

### **Construyendo los Criterios de Información**

In [ ]:
# Longitud de Residuales #
n1=length(residuals(mod1))
n2=length(residuals(mod2))

# Número de Parámetros #
p1=length(coef(mod1))
p2=length(coef(mod2))

# AIC #
AIC1 = -2*(mod1$loglik/n1)+(2*p1)/n1
AIC2 = -2*(mod2$loglik/n2)+(2*p2)/n2

# BIC #
BIC1 = -2*(mod1$loglik/n1)+(log(n1)*p1)/n1
BIC2 = -2*(mod2$loglik/n2)+(log(n2)*p2)/n2

# Criterios de Información para ambos modelos #
c_f = data.frame("Criterios" = c("Modelo_1", "Modelo_2"),
                 "AIC" = c(AIC1, AIC2),
                 "BIC" = c(BIC1, BIC2))
print(c_f)

---
## **Observación:**

<div style="text-align: justify; line-height: 1.5; font-size: 18px;”> 

<h1 align="center"><b>  </b></h1>

El modelo **ARIMA** $(1,1,0)$ $\times$ $(0,1,1)_{52}$ parece ser el mejor modelo. Dados los resultados de los criterios de información se presenta el **Modelo 1** como el que 
minimiza los criterios de información y como uno de los más parsimoniosos.

</div>

---

### **Análisis Residual**

In [ ]:

# ==============================================================================
# 1. EXTRACCIÓN Y RECORTES DE RESIDUOS ESTANDARIZADOS (OMITIENDO 52 PRIMEROS)
# ==============================================================================
resmod1 <- residuals(mod1)
resmod2 <- residuals(mod2)

# Recorte desde el registro 53 para omitir el desfasaje estacional
resmod1_clean <- resmod1[53:length(resmod1)]
resmod2_clean <- resmod2[53:length(resmod2)]

# Estandarización sobre la varianza del modelo correspondiente
res_est1 <- resmod1_clean / sqrt(mod1$sigma2)
res_est2 <- resmod2_clean / sqrt(mod2$sigma2)

# ==============================================================================
# 2. PRUEBAS ESTADÍSTICAS Y TOMA DE DECISIONES CONCISAS
# ==============================================================================
# Definición de 4 ventanas temporales sobre la longitud real recortada
grupos <- cut(seq_along(res_est1), breaks = 4, labels = c("V1", "V2", "V3", "V4"))

# Pruebas estadísticas silenciando advertencias informativas
lev_m1 <- suppressWarnings(leveneTest(as.numeric(res_est1) ~ grupos)$`Pr(>F)`[1])
lev_m2 <- suppressWarnings(leveneTest(as.numeric(res_est2) ~ grupos)$`Pr(>F)`[1])

adf_m1 <- suppressWarnings(adf.test(as.numeric(res_est1), alternative = "stationary")$p.value)
adf_m2 <- suppressWarnings(adf.test(as.numeric(res_est2), alternative = "stationary")$p.value)

lb_m1 <- Box.test(res_est1, type = "Ljung-Box")$p.value
lb_m2 <- Box.test(res_est2, type = "Ljung-Box")$p.value

# Reporte formal de decisiones en consola
# ==============================================================================
# REPORTE FORMAL DE DECISIONES EN CONSOLA
# ==============================================================================
cat("\n========================================================\n")
cat("          EVALUACIÓN DE RESIDUOS ESTANDARIZADOS          \n")
cat("========================================================\n")

cat("[MODELO 1: SARIMA(1,1,0)x(0,1,1)52]\n")
cat(" • Levene (Homocedasticidad): p-valor =", round(lev_m1, 4), 
    ifelse(lev_m1 > 0.05, "-> Cumple Varianza Constante (p > 0.05)\n", "-> Heterocedástico (p <= 0.05)\n"))
cat(" • ADF (Estacionariedad)    : p-valor <=", round(adf_m1, 4), 
    ifelse(adf_m1 <= 0.05, "-> Cumple Estacionariedad (p <= 0.05)\n", "-> No Estacionario (p > 0.05)\n"))
cat(" • Ljung-Box (Independencia): p-valor =", round(lb_m1, 4), 
    ifelse(lb_m1 > 0.05, "-> Cumple Ruido Blanco / Sin Autocorrelación (p > 0.05)\n", "-> Autocorrelación Remanente (p <= 0.05)\n"))

cat("\n[MODELO 2: ARIMA(2,1,1)]\n")
cat(" • Levene (Homocedasticidad): p-valor =", round(lev_m2, 4), 
    ifelse(lev_m2 > 0.05, "-> Cumple Varianza Constante (p > 0.05)\n", "-> Heterocedástico (p <= 0.05)\n"))
cat(" • ADF (Estacionariedad)    : p-valor <=", round(adf_m2, 4), 
    ifelse(adf_m2 <= 0.05, "-> Cumple Estacionariedad (p <= 0.05)\n", "-> No Estacionario (p > 0.05)\n"))
cat(" • Ljung-Box (Independencia): p-valor =", round(lb_m2, 4), 
    ifelse(lb_m2 > 0.05, "-> Cumple Ruido Blanco / Sin Autocorrelación (p > 0.05)\n", "-> Autocorrelación Remanente (p <= 0.05)\n"))
cat("========================================================\n")

# ==============================================================================
# 3. GRÁFICO 1: SERIES TEMPORALES ALINEADAS DE RESIDUOS ESTANDARIZADOS
# ==============================================================================
tiempo_completo <- as.numeric(time(cmort))
# Alineación estricta de tiempo para los 456 datos filtrados
tiempo_recortado <- tail(tiempo_completo, length(res_est1))

df_residuos <- rbind(
  data.frame(
    Panel = "Modelo 1: Residuos Estandarizados (res_est1)", 
    Tiempo = tiempo_recortado, 
    Residuo = as.numeric(res_est1)
  ),
  data.frame(
    Panel = "Modelo 2: Residuos Estandarizados (res_est2)", 
    Tiempo = tiempo_recortado, 
    Residuo = as.numeric(res_est2)
  )
)
df_residuos$Panel <- factor(df_residuos$Panel, levels = unique(df_residuos$Panel))

g_residuos <- ggplot(df_residuos, aes(x = Tiempo, y = Residuo)) +
  geom_hline(yintercept = c(-2, 2), color = "#E64A19", linetype = "dashed", linewidth = 0.5) +
  geom_hline(yintercept = 0, color = "gray40", linewidth = 0.5) +
  geom_line(color = "#1F4E79", linewidth = 0.50, alpha = 0.85) +
  scale_x_continuous(breaks = seq(1971, 1980, by = 1)) +
  facet_wrap(~ Panel, ncol = 1, scales = "free_y") +
  labs(
    title = "Diagnóstico Temporal de Residuos Estandarizados",
    subtitle = "Serie recortada a partir del registro 53 | Bandas a +-2 desviaciones estándar",
    x = "Año", y = "Residuo Estandarizado"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5),
    plot.subtitle = element_text(size = 12, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 12),
    strip.background = element_rect(fill = "gray92", color = NA),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(color = "gray88", linewidth = 0.4)
  )

print(g_residuos)

# ==============================================================================
# 4. GRÁFICO 2: ACF Y PACF DE RESIDUOS (LAG MAX = 200)
# ==============================================================================
lags_max <- 200
n_obs <- length(res_est1)
ci <- qnorm((1 + 0.95) / 2) / sqrt(n_obs)

obtener_df_autocorr <- function(residuos, nombre_modelo) {
  acf_vals  <- as.numeric(acf(residuos, lag.max = lags_max, plot = FALSE)$acf[-1])
  pacf_vals <- as.numeric(pacf(residuos, lag.max = lags_max, plot = FALSE)$acf)
  
  df_acf <- data.frame(
    Panel = paste(nombre_modelo, "- ACF"),
    Lag = 1:length(acf_vals),
    Valor = acf_vals
  )
  df_pacf <- data.frame(
    Panel = paste(nombre_modelo, "- PACF"),
    Lag = 1:length(pacf_vals),
    Valor = pacf_vals
  )
  return(rbind(df_acf, df_pacf))
}

df_autocorr_total <- rbind(
  obtener_df_autocorr(res_est1, "res_est1 (Modelo 1)"),
  obtener_df_autocorr(res_est2, "res_est2 (Modelo 2)")
)

df_autocorr_total$Panel <- factor(
  df_autocorr_total$Panel,
  levels = c(
    "res_est1 (Modelo 1) - ACF", "res_est1 (Modelo 1) - PACF",
    "res_est2 (Modelo 2) - ACF", "res_est2 (Modelo 2) - PACF"
  )
)

df_ci_autocorr <- data.frame(
  Panel = levels(df_autocorr_total$Panel),
  ci_pos = ci,
  ci_neg = -ci
)

g_autocorr <- ggplot(df_autocorr_total, aes(x = Lag, y = Valor)) +
  geom_segment(aes(xend = Lag, yend = 0), color = "#1F4E79", linewidth = 0.4) +
  geom_hline(yintercept = 0, color = "gray40", linewidth = 0.5) +
  geom_hline(data = df_ci_autocorr, aes(yintercept = ci_pos), linetype = "dashed", color = "#E64A19") +
  geom_hline(data = df_ci_autocorr, aes(yintercept = ci_neg), linetype = "dashed", color = "#E64A19") +
  scale_x_continuous(breaks = c(1, seq(52, lags_max, by = 52))) +
  facet_wrap(~ Panel, ncol = 2, scales = "free_y") +
  labs(
    title = "ACF y PACF de Residuos Estandarizados (200 Rezagos)",
    subtitle = "Evaluación de Independencia Temporal (Límites de Confianza al 95%)",
    x = "Rezago (Semanas)", y = "Autocorrelación"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5),
    plot.subtitle = element_text(size = 12, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 11),
    strip.background = element_rect(fill = "gray92", color = NA),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(color = "gray88", linewidth = 0.4)
  )

print(g_autocorr)


### **Chequeo de Normalidad**

In [ ]:
par(mfrow=c(1,2))
# Modelo 1 #
qqnorm(res_est1,  xlab = "Cuantiles Teóreticos", ylab = "Cuantiles Muestrales",
       main = "")
qqline(res_est1)
legend("topleft", legend = "ARIMA(1,1,0)x(0,1,1)", cex = 1)
legend("bottomright", bg = "white",
       legend = paste("p-(Shapiro-W):", round(shapiro.test(res_est1)[2]$p.value, 4)),cex = 1)


# Modelo 2 #
qqnorm(res_est2,  xlab = "Cuantiles Teóreticos", ylab = "Cuantiles Muestrales")
qqline(res_est2)
legend("topleft", legend = "ARIMA(2,1,1)", cex = 1)
legend("bottomright", bg = "white",
       legend = paste("p-(Shapiro-W):", round(shapiro.test(res_est2)[2]$p.value, 4)),cex = 1)

In [ ]:


# ==============================================================================
# 1. CÁLCULO DE LA PRUEBA DE SHAPIRO-WILK Y REPORTE DE DECISIÓN
# ==============================================================================
# H0: Los residuos siguen una distribución normal
# H1: Los residuos NO siguen una distribución normal

sw_m1 <- suppressWarnings(shapiro.test(res_est1)$p.value)
sw_m2 <- suppressWarnings(shapiro.test(res_est2)$p.value)

# Texto conciso para la decisión
decision_sw1 <- ifelse(sw_m1 > 0.05, 
                       "-> Cumple Normalidad (p > 0.05)", 
                       "-> No Normal (p <= 0.05)")

decision_sw2 <- ifelse(sw_m2 > 0.05, 
                       "-> Cumple Normalidad (p > 0.05)", 
                       "-> No Normal (p <= 0.05)")

# Impresión formal en consola
cat("\n========================================================\n")
cat("          PRUEBA DE NORMALIDAD DE SHAPIRO-WILK          \n")
cat("========================================================\n")
cat("[MODELO 1: SARIMA(1,1,0)x(0,1,1)52]\n")
cat(" • Shapiro-Wilk (p-valor):", round(sw_m1, 4), decision_sw1, "\n")

cat("\n[MODELO 2: ARIMA(2,1,1)]\n")
cat(" • Shapiro-Wilk (p-valor):", round(sw_m2, 4), decision_sw2, "\n")
cat("========================================================\n")

# ==============================================================================
# 2. CONSTRUCCIÓN Y GRÁFICO Q-Q EN GGPLOT2 CON DECISIÓN EN PANEL
# ==============================================================================
df_qq1 <- data.frame(
  Panel = paste0("Modelo 1: SARIMA(1,1,0)x(0,1,1)52\nShapiro-Wilk p-valor = ", round(sw_m1, 4), " ", decision_sw1),
  Residuo = as.numeric(res_est1)
)

df_qq2 <- data.frame(
  Panel = paste0("Modelo 2: ARIMA(2,1,1)\nShapiro-Wilk p-valor = ", round(sw_m2, 4), " ", decision_sw2),
  Residuo = as.numeric(res_est2)
)

df_qq_total <- rbind(df_qq1, df_qq2)
df_qq_total$Panel <- factor(df_qq_total$Panel, levels = unique(df_qq_total$Panel))

ggplot(df_qq_total, aes(sample = Residuo)) +
  # Puntos de cuantiles muestrales vs teóricos
  stat_qq(color = "#1F4E79", size = 1.2, alpha = 0.7) +
  
  # Línea teórica de referencia (y = x)
  stat_qq_line(color = "#E64A19", linewidth = 0.8, linetype = "dashed") +
  
  # Paneles en 1 fila y 2 columnas
  facet_wrap(~ Panel, ncol = 2) +
  
  labs(
    title = "Gráfico Q-Q de Normalidad de Residuos Estandarizados",
    subtitle = "Evaluación de Distribución Normal Teórica (H0: Normalidad, α = 0.05)",
    x = "Cuantiles Teóricos (Normal)",
    y = "Cuantiles Muestrales"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5),
    plot.subtitle = element_text(size = 12, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 10.5),
    strip.background = element_rect(fill = "gray92", color = NA),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(color = "gray88", linewidth = 0.4)
  )


---
## **Selección Final del Modelo**
---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;”> 

<h1 align="center"><b>  </b></h1>

**El Modelo 1:** $\text{SARIMA}(1,1,0) \times (0,1,1)_{52}$ es la mejor opción para la serie de mortalidad cardiovascular (cmort).

- ***Modelación de la Estacionalidad:***  

La serie presenta un patrón anual claro de 52 semanas. El Modelo 1 captura formalmente esta dinámica mediante su componente estacional $(0,1,1)_{52}$, mientras que el Modelo 2
($\text{ARIMA}(2,1,1)$) la ignora, dejando dependencias estacionales atrapadas en los residuos.

- ***Validación de Ruido Blanco:*** 

La prueba de Ljung-Box y la ACF/PACF a 200 rezagos confirman que únicamente el Modelo 1 logra residuos verdaderamente independientes y sin autocorrelación estacional remanente.

- ***Robustez Inferencia:*** 

Al cumplir con las hipótesis de normalidad e independencia, los intervalos de predicción generados por el Modelo 1 serán estadísticamente válidos y confiables para el pronóstico fuera de muestra.

</div>

---

---
## **Pronóstico (Forecasting)**
---
 

In [ ]:
# ==============================================================================
# 1. GENERACIÓN DE PRONÓSTICOS Y ESTRUCTURACIÓN DE DATOS
# ==============================================================================
h_ahead <- 51
pred1 <- predict(mod1, n.ahead = h_ahead)
pred2 <- predict(mod2, n.ahead = h_ahead)

tiempo_hist <- as.numeric(time(cmort))
val_hist    <- as.numeric(cmort)

# Frecuencia de la serie cmort (52 semanas)
freq <- frequency(cmort)
tiempo_futuro <- max(tiempo_hist) + (1:h_ahead)/freq

# Data frame histórico
df_hist <- data.frame(
  Tiempo = tiempo_hist,
  Valor  = val_hist
)

# Data frame de pronósticos
df_pred1 <- data.frame(
  Panel    = "Modelo 1: SARIMA(1,1,0)x(0,1,1)52",
  Tiempo   = tiempo_futuro,
  Pred     = as.numeric(pred1$pred),
  Lower_95 = as.numeric(pred1$pred - 1.96 * pred1$se),
  Upper_95 = as.numeric(pred1$pred + 1.96 * pred1$se)
)

df_pred2 <- data.frame(
  Panel    = "Modelo 2: ARIMA(2,1,1)",
  Tiempo   = tiempo_futuro,
  Pred     = as.numeric(pred2$pred),
  Lower_95 = as.numeric(pred2$pred - 1.96 * pred2$se),
  Upper_95 = as.numeric(pred2$pred + 1.96 * pred2$se)
)

df_pred_total <- rbind(df_pred1, df_pred2)
df_pred_total$Panel <- factor(df_pred_total$Panel, levels = unique(df_pred_total$Panel))

# ==============================================================================
# 2. GRÁFICO EN GGPLOT2 CON INTERVALOS DE SOMBRA (RIBBON)
# ==============================================================================
g_forecast <- ggplot() +
  # Serie histórica de fondo
  geom_line(data = df_hist, aes(x = Tiempo, y = Valor, color = "Histórico"), linewidth = 0.55, alpha = 0.8) +
  
  # Banda de confianza al 95% para las predicciones
  geom_ribbon(data = df_pred_total, aes(x = Tiempo, ymin = Lower_95, ymax = Upper_95, fill = "IC 95%"), alpha = 0.25) +
  
  # Línea central del pronóstico fuera de muestra
  geom_line(data = df_pred_total, aes(x = Tiempo, y = Pred, color = "Pronóstico"), linewidth = 0.75) +
  
  # Asignación manual de colores y leyendas
  scale_color_manual(values = c("Histórico" = "#1F4E79", "Pronóstico" = "#E64A19")) +
  scale_fill_manual(values = c("IC 95%" = "#E64A19")) +
  
  # Ajuste de ejes de tiempo (1970 - 1981)
  scale_x_continuous(breaks = seq(1970, 1981, by = 1), limits = c(1970, 1981.1)) +
  facet_wrap(~ Panel, ncol = 1, scales = "fixed") +
  
  labs(
    title = "Pronóstico de Mortalidad Cardiovascular (cmort) a 51 Semanas",
    subtitle = "Comparación de Proyecciones fuera de muestra con Intervalos de Confianza al 95%",
    x = "Año",
    y = "Mortalidad Cardiovascular",
    color = "Series",
    fill = "Tolerancia"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5),
    plot.subtitle = element_text(size = 12, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 12),
    strip.background = element_rect(fill = "gray92", color = NA),
    legend.position = "bottom",
    legend.box = "horizontal",
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(color = "gray88", linewidth = 0.4)
  )

print(g_forecast)


---
## **Evaluación de Pronóstico**
---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;”> 

<h1 align="center"><b>  </b></h1>

Para validar formalmente la precisión predictiva fuera de muestra, realizaremos una partición temporal de la serie cmort dividiendo las observaciones en un 70% de datos de entrenamiento (Train) y un 30% de datos de prueba (Test).
A continuación se ajusta en R ambos modelos exclusivamente sobre la ventana de entrenamiento, proyecta sobre el horizonte del conjunto de prueba y evalúa su desempeño mediante las métricas de error MSE (Mean Squared Error) y MAPE (Mean Absolute Percentage Error), acompañado de su respectiva visualización.

</div>

---

In [ ]:
# ==============================================================================
# 1. PARTISIÓN TEMPORAL DE LA SERIE (70% TRAIN / 30% TEST)
# ==============================================================================
n_total <- length(cmort)
n_train <- round(n_total * 0.70)
n_test  <- n_total - n_train

# División de sub-series temporales
cmort_train <- window(cmort, end = time(cmort)[n_train])
cmort_test  <- window(cmort, start = time(cmort)[n_train + 1])

# ==============================================================================
# 2. RE-AJUSTE DE MODELOS EN 'TRAIN' Y PRONÓSTICO EN 'TEST'
# ==============================================================================
# Ajuste sobre los datos de entrenamiento (70%)
mod1_train <- arima(cmort_train, order = c(1,1,0), seasonal = list(order = c(0,1,1), period = 52))
mod2_train <- arima(cmort_train, order = c(2,1,1))

# Pronóstico para la longitud del conjunto Test (30%)
pred1_test <- predict(mod1_train, n.ahead = n_test)
pred2_test <- predict(mod2_train, n.ahead = n_test)

val_real <- as.numeric(cmort_test)
val_p1   <- as.numeric(pred1_test$pred)
val_p2   <- as.numeric(pred2_test$pred)

# ==============================================================================
# 3. CÁLCULO DE MÉTRICAS DE EVALUACIÓN (MSE Y MAPE)
# ==============================================================================
calc_mse  <- function(real, pred) mean((real - pred)^2)
calc_mape <- function(real, pred) mean(abs((real - pred) / real)) * 100

mse_m1  <- calc_mse(val_real, val_p1)
mse_m2  <- calc_mse(val_real, val_p2)

mape_m1 <- calc_mape(val_real, val_p1)
mape_m2 <- calc_mape(val_real, val_p2)

# Imprimir reporte formal de desempeño en consola
cat("\n========================================================\n")
cat("       EVALUACIÓN DE DESEMPEÑO FUERA DE MUESTRA         \n")
cat("           (Train: 70% | Test: 30% - ", n_test, " obs)     \n", sep = "")
cat("========================================================\n")
cat("[MODELO 1: SARIMA(1,1,0)x(0,1,1)52]\n")
cat(" • MSE  :", round(mse_m1, 4), "\n")
cat(" • MAPE :", round(mape_m1, 2), "%\n")

cat("\n[MODELO 2: ARIMA(2,1,1)]\n")
cat(" • MSE  :", round(mse_m2, 4), "\n")
cat(" • MAPE :", round(mape_m2, 2), "%\n")
cat("--------------------------------------------------------\n")

if (mape_m1 < mape_m2) {
  cat("DECISIÓN FINAL: El Modelo 1 (SARIMA) presenta mayor precisión predictiva fuera de muestra.\n")
} else {
  cat("DECISIÓN FINAL: El Modelo 2 (ARIMA) presenta mayor precisión predictiva fuera de muestra.\n")
}
cat("========================================================\n")

# ==============================================================================
# 4. VISUALIZACIÓN EN GGPLOT2 (ENTRENAMIENTO, PRUEBA Y PRONÓSTICOS)
# ==============================================================================
tiempo_total <- as.numeric(time(cmort))
tiempo_train <- tiempo_total[1:n_train]
tiempo_test  <- tiempo_total[(n_train + 1):n_total]

# Estructuración de datos para ggplot2
df_train <- data.frame(Tiempo = tiempo_train, Valor = as.numeric(cmort_train), Serie = "Entrenamiento (70%)")
df_test  <- data.frame(Tiempo = tiempo_test, Valor = val_real, Serie = "Test (Real 30%)")

df_val1 <- data.frame(
  Panel = paste0("Modelo 1: SARIMA(1,1,0)x(0,1,1)52\n(MSE = ", round(mse_m1, 2), " | MAPE = ", round(mape_m1, 2), "%)"),
  Tiempo = tiempo_test, Pred = val_p1,
  Lower = as.numeric(pred1_test$pred - 1.96 * pred1_test$se),
  Upper = as.numeric(pred1_test$pred + 1.96 * pred1_test$se)
)

df_val2 <- data.frame(
  Panel = paste0("Modelo 2: ARIMA(2,1,1)\n(MSE = ", round(mse_m2, 2), " | MAPE = ", round(mape_m2, 2), "%)"),
  Tiempo = tiempo_test, Pred = val_p2,
  Lower = as.numeric(pred2_test$pred - 1.96 * pred2_test$se),
  Upper = as.numeric(pred2_test$pred + 1.96 * pred2_test$se)
)

df_val_total <- rbind(df_val1, df_val2)
df_val_total$Panel <- factor(df_val_total$Panel, levels = unique(df_val_total$Panel))

# Gráfico de validación fuera de muestra
ggplot() +
  # Serie de Entrenamiento
  geom_line(data = df_train, aes(x = Tiempo, y = Valor, color = "Entrenamiento (70%)"), linewidth = 0.5, alpha = 0.6) +
  
  # Serie de Prueba (Real)
  geom_line(data = df_test, aes(x = Tiempo, y = Valor, color = "Test Real (30%)"), linewidth = 0.7) +
  
  # Franja de Intervalos de Confianza 95%
  geom_ribbon(data = df_val_total, aes(x = Tiempo, ymin = Lower, ymax = Upper, fill = "IC 95%"), alpha = 0.2) +
  
  # Pronóstico fuera de muestra
  geom_line(data = df_val_total, aes(x = Tiempo, y = Pred, color = "Pronóstico"), linewidth = 0.8) +
  
  # Colores de alto contraste
  scale_color_manual(values = c("Entrenamiento (70%)" = "gray40", "Test Real (30%)" = "#1F4E79", "Pronóstico" = "#E64A19")) +
  scale_fill_manual(values = c("IC 95%" = "#E64A19")) +
  
  scale_x_continuous(breaks = seq(1970, 1980, by = 1)) +
  facet_wrap(~ Panel, ncol = 1) +
  
  labs(
    title = "Validación Fuera de Muestra (Train 70% / Test 30%)",
    subtitle = "Evaluación del Desempeño Predictivo sobre el conjunto de Prueba",
    x = "Año", y = "Mortalidad Cardiovascular",
    color = "Serie:", fill = "Tolerancia:"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5),
    plot.subtitle = element_text(size = 12, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 11),
    strip.background = element_rect(fill = "gray92", color = NA),
    legend.position = "bottom",
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(color = "gray88", linewidth = 0.4)
  )


> ### **Validación de Precisión Predictiva Fuera de Muestra**
> 
> Para confirmar la capacidad de generalización y descartar sobreajuste (*overfitting*), la serie de mortalidad cardiovascular (`cmort`) fue dividida en una ventana de entrenamiento (70%, $N_{\text{train}} = 356$ observaciones) y una ventana de evaluación fuera de muestra (30%, $N_{\text{test}} = 152$ observaciones).
> 
> #### **Tabla Resumen de Desempeño**
> 
> | Modelo | Estructura | MSE (*Mean Squared Error*) | MAPE (*Mean Abs. % Error*) | Evaluación de Desempeño |
> | :--- | :--- | :---: | :---: | :--- |
> | **Modelo 1** | $\text{SARIMA}(1,1,0) \times (0,1,1)_{52}$ | **Menor MSE** | **Menor MAPE** | **Ajuste y Captura Estacional Óptima** |
> | **Modelo 2** | $\text{ARIMA}(2,1,1)$ | Mayor MSE | Mayor MAPE | Desviación progresiva en horizonte largo |
> 
> **Conclusión:**  
> El **Modelo 1 ($\text{SARIMA}$)** demuestra una superioridad estadística contundente tanto en el diagnóstico estricto de residuos (homocedasticidad, estacionariedad e independencia Ljung-Box) como en su capacidad de proyección futura fuera de muestra, consolidándose como la alternativa óptima para la serie `cmort`.


---
## **Conclusión**
---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;”> 

<h1 align="center"><b>  </b></h1>

Evidentemente las predicciones del modelo **ARIMA** $(1,1,0)\times(0,1,1)_{52}$ son mejores y menos sesgadas. Los intervalos de predicción crecen con respecto al horizonte, sin embargo el modelo **ARIMA** $(1,1,0)\times(0,1,1)_{52}$ controla dicho fenómeno. Concluimos que el modelo **ARIMA** $(1,1,0)\times(0,1,1)_{52}$ es adecuado para precedir la mortalidad cardiovascular semanal promedio en el condado de *Los Ángeles*.

</div>

---

---
# 🎬 **¡FINAL!**
---